# 06 — Test Multi-pass Pipeline End-to-End

Run the full pipeline (untrained) on a real dataset to verify shapes and behavior.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import torch
import time

from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.pipeline.multipass import MultiPassPipeline
from ms_zerogad.pipeline.aggregation import aggregate_scores, aggregate_scores_with_breakdown
from ms_zerogad.training.losses import multipass_total_loss
from ms_zerogad.evaluation.metrics import compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load Cora

In [ ]:
A_sp, X_sp, y_np = load_graph_dataset('/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/Cora.mat')
A = sparse_to_torch_dense(A_sp).to(device)
X = feature_to_torch(X_sp, dense=True).to(device)
y = torch.from_numpy(y_np).long().to(device)

n = X.shape[0]
print(f'Cora: n={n}, f={X.shape[1]}, anomalies={int(y.sum().item())}')

## Build pipeline

In [ ]:
pipeline = MultiPassPipeline(
    d_prime=8,
    d_hidden=64,
    d_latent=32,
    D_rff=50,
    d_svd=32,
    k_smoothing=1,
    sigma=1.0,
    tau=0.5,
).to(device)

n_params = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

## Forward pass (training mode)

In [ ]:
pipeline.train()

t0 = time.time()
scores_list, features_list, tracker, extras = pipeline(X, A, is_training=True)
t_forward = time.time() - t0

print(f'Forward time: {t_forward:.2f}s')
print()
print(f'Pass 1: scores shape={scores_list[0].shape}')
print(f'Pass 2: scores shape={scores_list[1].shape} (m_pass1={extras["m_pass1"]})')
print(f'Pass 3: scores shape={scores_list[2].shape} (m_pass2={extras["m_pass2"]})')
print(f'Tracker levels: {len(tracker)}')

## Inspect cluster sizes

In [ ]:
import numpy as np

for level in [1, 2]:
    membership = tracker.get_membership(level)
    sizes = [len(v) for v in membership.values()]
    print(f'Level {level}: {len(sizes)} clusters')
    print(f'  Cluster size: min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.2f}, median={np.median(sizes):.0f}')
    empty = sum(1 for s in sizes if s == 0)
    if empty > 0:
        print(f'  Empty clusters: {empty}')

## Backward pass — verify gradients flow

In [ ]:
Z_list = [extras['Z_1'], extras['Z_2'], extras['Z_3']]
total_loss, per_pass_losses = multipass_total_loss(features_list, Z_list)

print(f'Total loss: {total_loss.item():.4f}')
for i, l in enumerate(per_pass_losses, 1):
    print(f'  Pass {i} loss: {l.item():.4f}')

total_loss.backward()

n_grads = sum(1 for p in pipeline.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
n_total = sum(1 for p in pipeline.parameters())
print(f'\nParameters with non-zero gradient: {n_grads}/{n_total}')

## Aggregation and metrics (UNTRAINED model)

In [ ]:
with torch.no_grad():
    breakdown = aggregate_scores_with_breakdown(scores_list, tracker, n)

print('Per-pass AUROC (UNTRAINED — should be near 0.5):')
for key in ['pass1', 'pass2', 'pass3', 'final']:
    m = compute_metrics(breakdown[key].detach(), y)
    print(f'  {key:<10}: AUROC={m["auroc"]:.4f}, AUPRC={m["auprc"]:.4f}')